# Plot players seasons

In [ ]:
from meml_griz import fetch
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt

Get 2025-2026 season

In [ ]:
season = fetch.SeasonData.from_date_range(
    team=fetch.NbaTeams.MEM,
    start_date=datetime(2025, 10, 1),
    end_date=datetime(2026, 6, 1),
)

Plot players PPG

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)


# Get active games
active_games = season.season_df[
    (season.season_df["isActive"]) & (season.season_df.gameParticipationRate > 25)
].copy()
active_games["medianNetRating"] = active_games.groupby("playerSlug")[
    "netRating"
].transform("median")

# Calculate season metrics for each player
df = active_games.groupby("playerSlug", as_index=False).agg(
    meanNetRating=("netRating", "mean"),
    stdNetRating=("netRating", "std"),
    medianNetRating=("netRating", "median"),
)
# Sort dataframe
active_games = active_games.sort_values(by=["medianNetRating"], ignore_index=True)

# Plot sub-plots
ax = axes[0]
sns.lineplot(
    data=active_games,
    x="playerSlug",
    y="netRating",
    errorbar="sd",
    err_style="bars",
    err_kws={"capsize": 5},
    marker="s",
    markeredgecolor="none",
    sort=False,
    ax=ax,
)
ax.grid()
ax.tick_params(rotation=90)


for ycol, ax in zip(["stdNetRating", "medianNetRating"], axes[1:].ravel()):
    sns.barplot(
        data=df,
        x="playerSlug",
        y=ycol,
        ax=ax,
    )
    ax.set_axisbelow(True)
    ax.grid(axis="y")
    ax.tick_params(rotation=90)

plt.show()

In [ ]:
df[df["playerSlug"] == "ty-jerome"].filter(regex="netRating|minutes")